In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Attention
from tensorflow.keras.layers import Concatenate
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.text import \
    text_to_word_sequence
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import \
    preprocess_input
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.sequence import \
    pad_sequences
import PIL
import pickle
import gzip
import logging

In [2]:
EPOCHS = 20
BATCH_SIZE = 128
MAX_WORDS = 10000
READ_IMAGES = 90000
LAYER_SIZE = 256
EMBEDDING_WIDTH = 128
OOV_WORD = 'UNK'
PAD_INDEX = 0
OOV_INDEX = 1
START_INDEX = MAX_WORDS - 2
STOP_INDEX = MAX_WORDS - 1
MAX_LENGTH = 60
TRAINING_FILE_DIR = 'tf_data/feature_vectors/'
TEST_FILE_DIR = '../data/test_images/'
TEST_IMAGES = ['boat.jpg',
               'cat.jpg',
               'table.jpg',
               'bird.jpg']

In [3]:
#test dataset
TEST_FILE_DIR = './coco_dataset/test2014/'
TEST_IMAGE = ['cat.jpg']

In [4]:
def read_training_file(file_name, max_len):
    pickle_file = gzip.open(file_name, 'rb')
    image_dict = pickle.load(pickle_file)
    pickle_file.close()
    image_paths = []
    dest_word_sequences = []
    for i, key in enumerate(image_dict):
        if i == READ_IMAGES:
            break
        image_item = image_dict[key]
        image_paths.append(image_item[0])
        caption = image_item[1]
        word_sequence = text_to_word_sequence(caption)
        dest_word_sequence = word_sequence[0:max_len]
        dest_word_sequences.append(dest_word_sequence)
    return image_paths, dest_word_sequences

In [5]:
def tokenize(sequences):
    tokenizer = Tokenizer(num_words=MAX_WORDS-2,
                          oov_token=OOV_WORD)
    tokenizer.fit_on_texts(sequences)
    token_sequences = tokenizer.texts_to_sequences(sequences)
    return tokenizer, token_sequences

In [6]:
def tokens_to_words(tokenizer, seq):
    word_seq = []
    for index in seq:
        if index == PAD_INDEX:
            word_seq.append('PAD')
        elif index == OOV_INDEX:
            word_seq.append(OOV_WORD)
        elif index == START_INDEX:
            word_seq.append('START')
        elif index == STOP_INDEX:
            word_seq.append('STOP')
        else:
            word_seq.append(tokenizer.sequences_to_texts(
                [[index]])[0])
    print(word_seq)

In [7]:
image_paths, dest_seq = read_training_file(TRAINING_FILE_DIR \
    + 'caption_file.pickle.gz', MAX_LENGTH)
dest_tokenizer, dest_token_seq = tokenize(dest_seq)

In [8]:
image_paths

['COCO_train2014_000000057870.jpg',
 'COCO_train2014_000000384029.jpg',
 'COCO_train2014_000000222016.jpg',
 'COCO_train2014_000000520950.jpg',
 'COCO_train2014_000000069675.jpg',
 'COCO_train2014_000000547471.jpg',
 'COCO_train2014_000000122688.jpg',
 'COCO_train2014_000000392136.jpg',
 'COCO_train2014_000000398494.jpg',
 'COCO_train2014_000000090570.jpg',
 'COCO_train2014_000000504616.jpg',
 'COCO_train2014_000000161919.jpg',
 'COCO_train2014_000000457732.jpg',
 'COCO_train2014_000000044404.jpg',
 'COCO_train2014_000000004428.jpg',
 'COCO_train2014_000000170558.jpg',
 'COCO_train2014_000000405613.jpg',
 'COCO_train2014_000000283524.jpg',
 'COCO_train2014_000000037015.jpg',
 'COCO_train2014_000000071631.jpg',
 'COCO_train2014_000000491269.jpg',
 'COCO_train2014_000000365363.jpg',
 'COCO_train2014_000000064460.jpg',
 'COCO_train2014_000000581674.jpg',
 'COCO_train2014_000000470072.jpg',
 'COCO_train2014_000000344806.jpg',
 'COCO_train2014_000000084427.jpg',
 'COCO_train2014_00000031723

In [9]:
dest_seq

[['a', 'restaurant', 'has', 'modern', 'wooden', 'tables', 'and', 'chairs'],
 ['a',
  'man',
  'preparing',
  'desserts',
  'in',
  'a',
  'kitchen',
  'covered',
  'in',
  'frosting'],
 ['a',
  'big',
  'red',
  'telephone',
  'booth',
  'that',
  'a',
  'man',
  'is',
  'standing',
  'in'],
 ['the', 'kitchen', 'is', 'full', 'of', 'spices', 'on', 'the', 'rack'],
 ['a', 'child', 'and', 'woman', 'are', 'cooking', 'in', 'the', 'kitchen'],
 ['a',
  'black',
  'and',
  'white',
  'image',
  'of',
  'a',
  'man',
  'in',
  'a',
  'suit',
  'wearing',
  'glasses',
  'walking',
  'through',
  'a',
  'door'],
 ['the',
  'huge',
  'clock',
  'on',
  'the',
  'wall',
  'is',
  'near',
  'a',
  'wooden',
  'table'],
 ['a', 'large', 'bus', 'and', 'some', 'people', 'on', 'the', 'street'],
 ['a',
  'bicycle',
  'parked',
  'in',
  'a',
  'kitchen',
  'with',
  'a',
  'stove',
  'and',
  'cabinets'],
 ['two',
  'people',
  'in',
  'a',
  'food',
  'truck',
  'one',
  'looking',
  'at',
  'an',
  'orde

In [10]:
dest_tokenizer

In [11]:
dest_token_seq

[[2, 384, 70, 629, 76, 547, 8, 286],
 [2, 10, 367, 1283, 6, 2, 62, 79, 6, 968],
 [2, 181, 43, 1232, 1562, 26, 2, 10, 9, 13, 6],
 [5, 62, 9, 237, 3, 2724, 4, 5, 722],
 [2, 172, 8, 20, 22, 587, 6, 5, 62],
 [2, 40, 8, 17, 319, 3, 2, 10, 6, 2, 279, 84, 275, 50, 94, 2, 310],
 [5, 682, 89, 4, 5, 122, 9, 44, 2, 76, 23],
 [2, 27, 67, 8, 34, 18, 4, 5, 25],
 [2, 356, 66, 6, 2, 62, 7, 2, 271, 8, 444],
 [15, 18, 6, 2, 57, 119, 126, 88, 19, 16, 1927],
 [2, 31, 6, 17, 9, 13, 6, 2, 62],
 [2, 31, 9, 257, 2, 2470, 7, 2, 443, 8, 388],
 [2, 62, 7, 2, 23, 8, 34, 286],
 [2, 62, 70, 307, 444, 2, 1411, 109, 8, 239],
 [2, 1284, 367, 57, 146, 3, 2, 62, 44, 2, 120],
 [894, 301, 106, 554, 54, 12, 19, 509, 3084],
 [2, 29, 3, 73, 19, 2, 23, 367, 57, 160],
 [2, 10, 257, 36, 212, 4, 21, 3, 2, 57, 518],
 [1435, 22, 367, 57, 19, 2, 384, 145, 3085, 2277],
 [494, 46, 23, 334, 97, 2, 3988, 447, 7, 196],
 [2, 147, 832, 1140, 147, 578, 1928, 62],
 [15, 18, 72, 2, 153, 227, 1829, 131],
 [2, 62, 6, 2, 384, 7, 57, 4, 5, 167],

In [12]:
class ImageCaptionSequence(Sequence):
    def __init__(self, image_paths, dest_input_data,
                 dest_target_data, batch_size):
        self.image_paths = image_paths
        self.dest_input_data = dest_input_data
        self.dest_target_data = dest_target_data
        self.batch_size = batch_size

    def __len__(self):
        return int(np.ceil(len(self.dest_input_data) /
            float(self.batch_size)))

    def __getitem__(self, idx):
        batch_x0 = self.image_paths[
            idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_x1 = self.dest_input_data[
            idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_y = self.dest_target_data[
            idx * self.batch_size:(idx + 1) * self.batch_size]
        image_features = []
        for image_id in batch_x0:
            file_name = TRAINING_FILE_DIR \
                + image_id + '.pickle.gzip'
            pickle_file = gzip.open(file_name, 'rb')
            feature_vector = pickle.load(pickle_file)
            pickle_file.close()
            image_features.append(feature_vector)
        return [np.array(image_features),
                np.array(batch_x1)], np.array(batch_y)

In [13]:
dest_target_token_seq = [x + [STOP_INDEX] for x in dest_token_seq]
dest_input_token_seq = [[START_INDEX] + x for x in
                        dest_target_token_seq]
dest_input_data = pad_sequences(dest_input_token_seq,
                                padding='post')
dest_target_data = pad_sequences(   dest_target_token_seq, padding='post',
                                    maxlen=len(dest_input_data[0]))

In [14]:
image_sequence = ImageCaptionSequence(
    image_paths, dest_input_data, dest_target_data, BATCH_SIZE)

In [15]:
#Encoder
feature_vector_input = Input(shape=(14,14,512))

#Layers
enc_mean_layer = GlobalAveragePooling2D()
enc_layer_h = Dense(LAYER_SIZE)
enc_layer_c = Dense(LAYER_SIZE)

#Connect Layers
enc_mean_layer_output = enc_mean_layer(feature_vector_input)
enc_layer_h_outputs = enc_layer_h(enc_mean_layer_output)
enc_layer_c_outputs = enc_layer_c(enc_mean_layer_output)

#Output
enc_layer_outputs = [enc_layer_h_outputs, enc_layer_c_outputs]

#Build Model
enc_model_top = Model(feature_vector_input, enc_layer_outputs)
enc_model_top.summary()

Instructions for updating:
If using Keras pass *_constraint arguments to layers.
Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 14, 14, 512) 0                                            
__________________________________________________________________________________________________
global_average_pooling2d (Globa (None, 512)          0           input_1[0][0]                    
__________________________________________________________________________________________________
dense (Dense)                   (None, 256)          131328      global_average_pooling2d[0][0]   
__________________________________________________________________________________________________
dense_1 (Dense)                 (None, 256)          131328      global_average_pooling2d[0][0]   
Total params:

In [16]:
#Decoder
#Input
dec_feature_vector_input = Input(shape=(14, 14, 512))
dec_embedding_input = Input(shape=(None, ))
dec_layer1_state_input_h = Input(shape=(LAYER_SIZE,))
dec_layer1_state_input_c = Input(shape=(LAYER_SIZE,))

#Layer
dec_reshape_layer = Reshape((196, 512),
                            input_shape=(14, 14, 512,))
dec_attention_layer = Attention()
dec_query_layer = Dense(512)
dec_embedding_layer = Embedding(output_dim=EMBEDDING_WIDTH,
                                input_dim=MAX_WORDS,
                                mask_zero=False)
dec_layer1 = LSTM(LAYER_SIZE, return_state=True,
                  return_sequences=True)
dec_concat_layer = Concatenate()
dec_layer2 = Dense(MAX_WORDS, activation='softmax')

#Connect Layers
dec_embedding_layer_outputs = dec_embedding_layer(
    dec_embedding_input)
dec_reshape_layer_outputs = dec_reshape_layer(
    dec_feature_vector_input)
dec_layer1_outputs, dec_layer1_state_h, dec_layer1_state_c = \
    dec_layer1(dec_embedding_layer_outputs, initial_state=[
        dec_layer1_state_input_h, dec_layer1_state_input_c])
dec_query_layer_outputs = dec_query_layer(dec_layer1_outputs)
dec_attention_layer_outputs = dec_attention_layer(
    [dec_query_layer_outputs, dec_reshape_layer_outputs])
dec_layer2_inputs = dec_concat_layer(
    [dec_layer1_outputs, dec_attention_layer_outputs])
dec_layer2_outputs = dec_layer2(dec_layer2_inputs)

# Build the model.
dec_model = Model([dec_feature_vector_input,
                   dec_embedding_input,
                   dec_layer1_state_input_h,
                   dec_layer1_state_input_c],
                  [dec_layer2_outputs, dec_layer1_state_h,
                   dec_layer1_state_c])
dec_model.summary()

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Model: "model_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_3 (InputLayer)            [(None, None)]       0                                            
__________________________________________________________________________________________________
embedding (Embedding)           (None, None, 128)    1280000     input_3[0][0]                    
__________________________________________________________________________________________________
input_4 (InputLayer)            [(None, 256)]        0                                            
__________________________________________________________________________________________________
input_5 (InputLayer)            [(None, 256)]        0                   

In [17]:
#Compile Mddel
train_feature_vector_input = Input(shape=(14, 14, 512))
train_dec_embedding_input = Input(shape=(None, ))
intermediate_state = enc_model_top(train_feature_vector_input)
train_dec_output, _, _ = dec_model([train_feature_vector_input,
                                    train_dec_embedding_input] +
                                    intermediate_state)
training_model = Model([train_feature_vector_input,
                        train_dec_embedding_input],
                        [train_dec_output])
training_model.compile(loss='sparse_categorical_crossentropy',
                       optimizer='adam', metrics =['accuracy'])
training_model.summary()

Model: "model_2"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_6 (InputLayer)            [(None, 14, 14, 512) 0                                            
__________________________________________________________________________________________________
input_7 (InputLayer)            [(None, None)]       0                                            
__________________________________________________________________________________________________
model (Model)                   [(None, 256), (None, 262656      input_6[0][0]                    
__________________________________________________________________________________________________
model_1 (Model)                 [(None, None, 10000) 9495824     input_6[0][0]                    
                                                                 input_7[0][0]              

In [18]:
#Full Model for Inference
conv_model = VGG19(weights='imagenet')
conv_model_outputs = conv_model.get_layer('block5_conv4').output
intermediate_state = enc_model_top(conv_model_outputs)
inference_enc_model = Model([conv_model.input],
                            intermediate_state
                            + [conv_model_outputs])
inference_enc_model.summary()

Model: "model_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_8 (InputLayer)         [(None, 224, 224, 3)]     0         
_________________________________________________________________
block1_conv1 (Conv2D)        (None, 224, 224, 64)      1792      
_________________________________________________________________
block1_conv2 (Conv2D)        (None, 224, 224, 64)      36928     
_________________________________________________________________
block1_pool (MaxPooling2D)   (None, 112, 112, 64)      0         
_________________________________________________________________
block2_conv1 (Conv2D)        (None, 112, 112, 128)     73856     
_________________________________________________________________
block2_conv2 (Conv2D)        (None, 112, 112, 128)     147584    
_________________________________________________________________
block2_pool (MaxPooling2D)   (None, 56, 56, 128)       0   

In [19]:
for i in range(EPOCHS): # Train and evaluate model
    print('step: ' , i)
    history = training_model.fit(image_sequence, epochs=1)
    for filename in TEST_IMAGES:
        # Determine dimensions.
        image = load_img(TEST_FILE_DIR + filename)
        width = image.size[0]
        height = image.size[1]

        # Resize so shortest side is 256 pixels.
        if height > width:
            image = load_img(
                TEST_FILE_DIR + filename,
                target_size=(int(height/width*256), 256))
        else:
            image = load_img(
                TEST_FILE_DIR + filename,
                target_size=(256, int(width/height*256)))
        width = image.size[0]
        height = image.size[1]
        image_np = img_to_array(image)

        # Crop to center 224x224 region.
        h_start = int((height-224)/2)
        w_start = int((width-224)/2)
        image_np = image_np[h_start:h_start+224,
                            w_start:w_start+224]

        # Run image through encoder.
        image_np = np.expand_dims(image_np, axis=0)
        x = preprocess_input(image_np)
        dec_layer1_state_h, dec_layer1_state_c, feature_vector = \
            inference_enc_model.predict(x, verbose=0)

        # Predict sentence word for word.
        prev_word_index = START_INDEX
        produced_string = ''
        pred_seq = []
        for j in range(MAX_LENGTH):
            x = np.reshape(np.array(prev_word_index), (1, 1))
            preds, dec_layer1_state_h, dec_layer1_state_c = \
                dec_model.predict(
                    [feature_vector, x, dec_layer1_state_h,
                     dec_layer1_state_c], verbose=0)
            prev_word_index = np.asarray(preds[0][0]).argmax()
            pred_seq.append(prev_word_index)
            if prev_word_index == STOP_INDEX:
                break
        tokens_to_words(dest_tokenizer, pred_seq)
        print('\n\n')

step:  0
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


InvalidArgumentError: 2 root error(s) found.
  (0) Invalid argument: Can not squeeze dim[1], expected a dimension of 1, got 51
	 [[{{node metrics/acc/Squeeze}}]]
	 [[training/Adam/Pow_1/_377]]
  (1) Invalid argument: Can not squeeze dim[1], expected a dimension of 1, got 51
	 [[{{node metrics/acc/Squeeze}}]]
0 successful operations.
0 derived errors ignored.